# ResNet50 End-to-End Transfer Learning

This notebook implements ResNet50 transfer learning for four-class brain tumour MRI classification using the predefined five-fold cross-validation assignments.

In [1]:
import sys
import tensorflow as tf

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nTensorFlow version:")
print(tf.__version__)

print("\nAvailable GPUs:")
print(tf.config.list_physical_devices("GPU"))

Python executable:
c:\Users\kwsta\venvs\resnet50\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

TensorFlow version:
2.21.0

Available GPUs:
[]


In [ ]:
# Import TensorFlow and use the shorter name "tf".
import tensorflow as tf

# Import the prebuilt ResNet50 architecture from TensorFlow/Keras.
from tensorflow.keras.applications import ResNet50


# Print the installed TensorFlow version.
print("TensorFlow version:", tf.__version__)

# Print a progress message before loading the model.
print("Starting ResNet50 loading...")


try:
    # Create the ResNet50 base model.
    resnet50_base = ResNet50(
        # Load weights that were previously trained on the ImageNet dataset.
        weights="imagenet",

        # Remove ResNet50's original classifier, which predicts
        # the 1,000 ImageNet classes.
        # We keep only the feature-extraction part of the network.
        include_top=False,

        # ResNet50 expects images that are:
        # 224 pixels high, 224 pixels wide, with 3 RGB channels.
        input_shape=(224, 224, 3),

        # Apply global average pooling to the final feature maps.
        # This produces one 2,048-value feature vector per image.
        pooling="avg"
    )

    # Freeze the complete pretrained ResNet50 model.
    # Its ImageNet weights will not be changed during training.
    # The model will be used only as a fixed feature extractor.
    resnet50_base.trainable = False

    print("\nResNet50 loaded successfully.")

    # Print the internal name assigned to the model.
    print("Model name:", resnet50_base.name)

    # Print the expected input shape.
    # None represents the batch size, which can vary.
    print("Input shape:", resnet50_base.input_shape)

    # Print the output shape.
    # With pooling="avg", each image produces 2,048 features.
    print("Output shape:", resnet50_base.output_shape)

    # Count all individual numerical parameters in the model.
    # This includes both trainable and non-trainable parameters.
    print(
        "Total parameters:",
        resnet50_base.count_params()
    )

    # Count the variable tensors that can be updated during training.
    # This should be 0 because the complete model was frozen.
    print(
        "Trainable variables:",
        len(resnet50_base.trainable_variables)
    )

    # Count the variable tensors that remain fixed.
    # One variable may contain thousands or millions of parameters.
    print(
        "Non-trainable variables:",
        len(resnet50_base.non_trainable_variables)
    )


except Exception as error:
    # Run this section if ResNet50 cannot be loaded.
    # Possible causes include download, memory, or configuration problems.
    print("\nResNet50 loading failed.")

    # Print the name of the error type.
    print("Error type:", type(error).__name__)

    # Print the detailed error message.
    print("Error message:", error)

TensorFlow version: 2.21.0
Starting ResNet50 loading...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step

ResNet50 loaded successfully.
Model name: resnet50
Input shape: (None, 224, 224, 3)
Output shape: (None, 2048)
Total parameters: 23587712
Trainable variables: 0
Non-trainable variables: 318


In [4]:
# Reproducible ResNet50 experiment setup

import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ---------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


# Request deterministic TensorFlow operations where supported
try:
    tf.config.experimental.enable_op_determinism()
    determinism_enabled = True
except Exception as error:
    determinism_enabled = False
    determinism_error = str(error)


# ---------------------------------------------------------
# 2. Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

FOLDS_FILE = (
    PROJECT_ROOT
    / "splits"
    / "five_fold_cross_validation.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "resnet50_transfer_learning"
)


# ---------------------------------------------------------
# 3. Initial experiment configuration
# ---------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

BATCH_SIZE = 16

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]


# ---------------------------------------------------------
# 4. Display configuration
# ---------------------------------------------------------

print("--- Environment ---")
print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

print("\n--- Reproducibility ---")
print("Random seed:", RANDOM_SEED)
print("Deterministic operations enabled:", determinism_enabled)

if not determinism_enabled:
    print("Determinism error:", determinism_error)

print("\n--- Project paths ---")
print("Project root:", PROJECT_ROOT)
print("Dataset folder:", DATA_DIR)
print("Dataset folder exists:", DATA_DIR.exists())
print("Fold file:", FOLDS_FILE)
print("Fold file exists:", FOLDS_FILE.exists())

print("\n--- Experiment configuration ---")
print(
    "Input shape:",
    (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        IMAGE_CHANNELS
    )
)
print("Number of classes:", NUMBER_OF_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Class names:", CLASS_NAMES)

--- Environment ---
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\kwsta\venvs\resnet50\Scripts\python.exe
TensorFlow version: 2.21.0
Keras version: 3.15.1

--- Reproducibility ---
Random seed: 42
Deterministic operations enabled: True

--- Project paths ---
Project root: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification
Dataset folder: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\processed_data_cropped
Dataset folder exists: True
Fold file: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\splits\five_fold_cross_validation.csv
Fold file exists: True

--- Experiment configuration ---
Input shape: (224, 224, 3)
Number of classes: 4
Batch size: 16
Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [5]:
# Load and validate the fixed five-fold assignments

folds_df = pd.read_csv(FOLDS_FILE)


# ---------------------------------------------------------
# 1. Validate the table structure
# ---------------------------------------------------------

required_columns = {
    "relative_path",
    "class",
    "fold"
}

missing_columns = (
    required_columns
    - set(folds_df.columns)
)

if missing_columns:
    raise ValueError(
        "The fold file is missing columns: "
        f"{sorted(missing_columns)}"
    )


# ---------------------------------------------------------
# 2. Create complete image paths
# ---------------------------------------------------------

folds_df["image_path"] = (
    folds_df["relative_path"]
    .apply(lambda path: DATA_DIR / Path(path))
)


# ---------------------------------------------------------
# 3. Validate paths and assignments
# ---------------------------------------------------------

missing_image_paths = [
    image_path
    for image_path in folds_df["image_path"]
    if not image_path.exists()
]

unexpected_classes = sorted(
    set(folds_df["class"])
    - set(CLASS_NAMES)
)

unexpected_folds = sorted(
    set(folds_df["fold"])
    - {1, 2, 3, 4, 5}
)

duplicate_paths = (
    folds_df["relative_path"]
    .duplicated()
    .sum()
)

missing_values = (
    folds_df[
        [
            "relative_path",
            "class",
            "fold"
        ]
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# 4. Display the validation results
# ---------------------------------------------------------

print("--- Fold file ---")
print("Rows:", len(folds_df))
print("Columns:", folds_df.columns.tolist())

print("\n--- Assignment checks ---")
print("Duplicate relative paths:", duplicate_paths)
print("Missing image files:", len(missing_image_paths))
print("Unexpected classes:", unexpected_classes)
print("Unexpected fold values:", unexpected_folds)

print("\nMissing values:")
print(missing_values)

print("\n--- Images per fold and class ---")

fold_distribution = pd.crosstab(
    folds_df["fold"],
    folds_df["class"]
)

fold_distribution["total"] = (
    fold_distribution.sum(axis=1)
)

display(fold_distribution)

print("\n--- Sample records ---")

display(
    folds_df[
        [
            "relative_path",
            "class",
            "fold",
            "image_path"
        ]
    ].head()
)


# ---------------------------------------------------------
# 5. Stop if an important validation check failed
# ---------------------------------------------------------

if len(folds_df) != 5600:
    raise ValueError(
        "Expected 5,600 Training images, "
        f"but found {len(folds_df)}."
    )

if duplicate_paths != 0:
    raise ValueError(
        "Duplicate paths were found in the fold file."
    )

if missing_image_paths:
    raise FileNotFoundError(
        f"{len(missing_image_paths)} image files are missing."
    )

if unexpected_classes:
    raise ValueError(
        f"Unexpected classes found: {unexpected_classes}"
    )

if unexpected_folds:
    raise ValueError(
        f"Unexpected fold values found: {unexpected_folds}"
    )

if missing_values.sum() != 0:
    raise ValueError(
        "Missing values were found in the fold file."
    )

print("\nFold-assignment validation passed.")

--- Fold file ---
Rows: 5600
Columns: ['relative_path', 'class', 'fold', 'image_path']

--- Assignment checks ---
Duplicate relative paths: 0
Missing image files: 0
Unexpected classes: []
Unexpected fold values: []

Missing values:
relative_path    0
class            0
fold             0
dtype: int64

--- Images per fold and class ---


class,glioma,meningioma,notumor,pituitary,total
fold,,,,,
1,280,280,280,280,1120
2,280,280,280,280,1120
3,280,280,280,280,1120
4,280,280,280,280,1120
5,280,280,280,280,1120



--- Sample records ---


,relative_path,class,fold,image_path
0,Training/glioma/Tr-gl_100.png,glioma,1,c:\Users\kwsta\OneDrive\Desktop\Desetation pro...
1,Training/glioma/Tr-gl_1001.png,glioma,1,c:\Users\kwsta\OneDrive\Desktop\Desetation pro...
2,Training/glioma/Tr-gl_1003.png,glioma,1,c:\Users\kwsta\OneDrive\Desktop\Desetation pro...
3,Training/glioma/Tr-gl_1014.png,glioma,1,c:\Users\kwsta\OneDrive\Desktop\Desetation pro...
4,Training/glioma/Tr-gl_1015.png,glioma,1,c:\Users\kwsta\OneDrive\Desktop\Desetation pro...



Fold-assignment validation passed.


In [ ]:
# Test loading one grayscale PNG for ResNet50

from tensorflow.keras.applications.resnet50 import (
    preprocess_input
)


# ---------------------------------------------------------
# 1. Select one sample image
# ---------------------------------------------------------

# Selecting the first row from the folds_df which is the dataframe with all the preprocessed images.
sample_record = folds_df.iloc[0]

# Image path
sample_image_path = sample_record["image_path"]

# Image class
sample_class_name = sample_record["class"]

# Image fold
sample_fold = sample_record["fold"]


# ---------------------------------------------------------
# 2. Read the stored single-channel PNG
# ---------------------------------------------------------

# Reading the raw binary content of the image
image_bytes = tf.io.read_file(
    str(sample_image_path)
)

# Reconstructing the pixels
grayscale_image = tf.io.decode_png(
    image_bytes,
    channels=1
)


# ---------------------------------------------------------
# 3. Convert grayscale to three-channel pseudo-RGB
# ---------------------------------------------------------

# Converting the single-channel grayscale image into a three-channel image.
pseudo_rgb_image = tf.image.grayscale_to_rgb(
    grayscale_image
)

# Convert to float32 while retaining the 0–255 range
float_image = tf.cast(
    pseudo_rgb_image,
    tf.float32
)


# Apply the preprocessing required by ResNet50
preprocessed_image = preprocess_input(
    float_image
)


# ---------------------------------------------------------
# 4. Validate the result
# ---------------------------------------------------------

print("--- Sample record ---")
print("Path:", sample_image_path)
print("Class:", sample_class_name)
print("Fold:", sample_fold)

print("\n--- Stored grayscale image ---")
print("Shape:", grayscale_image.shape)
print("Data type:", grayscale_image.dtype)
print(
    "Pixel range:",
    int(tf.reduce_min(grayscale_image)),
    "to",
    int(tf.reduce_max(grayscale_image))
)

print("\n--- Three-channel pseudo-RGB image ---")
print("Shape:", pseudo_rgb_image.shape)
print("Data type:", pseudo_rgb_image.dtype)

channels_identical = (
    tf.reduce_all(
        pseudo_rgb_image[:, :, 0]
        ==
        pseudo_rgb_image[:, :, 1]
    )
    and
    tf.reduce_all(
        pseudo_rgb_image[:, :, 0]
        ==
        pseudo_rgb_image[:, :, 2]
    )
)

print(
    "All three channels identical:",
    bool(channels_identical)
)

print("\n--- ResNet50-preprocessed image ---")
print("Shape:", preprocessed_image.shape)
print("Data type:", preprocessed_image.dtype)
print(
    "Value range:",
    float(tf.reduce_min(preprocessed_image)),
    "to",
    float(tf.reduce_max(preprocessed_image))
)


# ---------------------------------------------------------
# 5. Stop if the input is not correct
# ---------------------------------------------------------

if grayscale_image.shape != (224, 224, 1):
    raise ValueError(
        "Expected stored image shape "
        "(224, 224, 1), but found "
        f"{grayscale_image.shape}."
    )

if pseudo_rgb_image.shape != (224, 224, 3):
    raise ValueError(
        "Expected ResNet50 input shape "
        "(224, 224, 3), but found "
        f"{pseudo_rgb_image.shape}."
    )

if not bool(channels_identical):
    raise ValueError(
        "The replicated image channels are not identical."
    )

print("\nSample image preprocessing passed.")

--- Sample record ---
Path: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\processed_data_cropped\Training\glioma\Tr-gl_100.png
Class: glioma
Fold: 1

--- Stored grayscale image ---
Shape: (224, 224, 1)
Data type: <dtype: 'uint8'>
Pixel range: 0 to 244

--- Three-channel pseudo-RGB image ---
Shape: (224, 224, 3)
Data type: <dtype: 'uint8'>
All three channels identical: True

--- ResNet50-preprocessed image ---
Shape: (224, 224, 3)
Data type: <dtype: 'float32'>
Value range: -123.68000030517578 to 140.06100463867188

Sample image preprocessing passed.


In [ ]:
# Create reusable label mapping and image-loading function


# ---------------------------------------------------------
# 1. Create a fixed class-to-index mapping
# ---------------------------------------------------------

# Converting each class name into a numerical label
CLASS_TO_INDEX = {
    class_name: class_index
    for class_index, class_name
    in enumerate(CLASS_NAMES)
}

# Converting each numerical label into a class
INDEX_TO_CLASS = {
    class_index: class_name
    for class_name, class_index
    in CLASS_TO_INDEX.items()
}


# Add the numeric label to the fold table
folds_df["class_index"] = (
    folds_df["class"]
    .map(CLASS_TO_INDEX)
)


# ---------------------------------------------------------
# 2. Validate the label mapping
# ---------------------------------------------------------

if folds_df["class_index"].isna().any():
    unknown_classes = sorted(
        folds_df.loc[
            folds_df["class_index"].isna(),
            "class"
        ].unique()
    )

    raise ValueError(
        f"Unknown class labels found: {unknown_classes}"
    )


folds_df["class_index"] = (
    folds_df["class_index"]
    .astype(np.int32)
)


# ---------------------------------------------------------
# 3. Define the reusable image-loading function
# ---------------------------------------------------------

def load_and_preprocess_resnet50_image(
    image_path,
    class_index
):
    """
    Load one stored grayscale PNG and prepare it for ResNet50.

    Parameters
    ----------
    image_path:
        Tensor containing the image path.

    class_index:
        Integer class label.

    Returns
    -------
    image:
        Float32 tensor with shape (224, 224, 3), processed
        using the ResNet50 preprocessing function.

    class_index:
        Integer class label.
    """

    image_bytes = tf.io.read_file(
        image_path
    )

    grayscale_image = tf.io.decode_png(
        image_bytes,
        channels=1
    )

    # Confirm the static stored-image dimensions
    grayscale_image = tf.ensure_shape(
        grayscale_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            1
        )
    )

    pseudo_rgb_image = tf.image.grayscale_to_rgb(
        grayscale_image
    )

    pseudo_rgb_image = tf.cast(
        pseudo_rgb_image,
        tf.float32
    )

    preprocessed_image = preprocess_input(
        pseudo_rgb_image
    )

    preprocessed_image = tf.ensure_shape(
        preprocessed_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS
        )
    )

    class_index = tf.cast(
        class_index,
        tf.int32
    )

    return preprocessed_image, class_index


# ---------------------------------------------------------
# 4. Test the reusable function
# ---------------------------------------------------------

test_record = folds_df.iloc[0]

test_image, test_label = (
    load_and_preprocess_resnet50_image(
        tf.constant(
            str(test_record["image_path"])
        ),
        tf.constant(
            test_record["class_index"]
        )
    )
)


print("--- Class mapping ---")

for class_index in sorted(INDEX_TO_CLASS):
    print(
        class_index,
        "->",
        INDEX_TO_CLASS[class_index]
    )


print("\n--- Label distribution ---")

display(
    folds_df[
        [
            "class",
            "class_index"
        ]
    ]
    .value_counts()
    .sort_index()
    .rename("number_of_images")
    .reset_index()
)


print("\n--- Reusable loader test ---")
print("Image shape:", test_image.shape)
print("Image data type:", test_image.dtype)
print("Numeric label:", int(test_label))
print(
    "Decoded class:",
    INDEX_TO_CLASS[int(test_label)]
)
print(
    "Expected class:",
    test_record["class"]
)
print(
    "All image values finite:",
    bool(
        tf.reduce_all(
            tf.math.is_finite(test_image)
        )
    )
)


# ---------------------------------------------------------
# 5. Final validation
# ---------------------------------------------------------

if test_image.shape != (
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    IMAGE_CHANNELS
):
    raise ValueError(
        f"Unexpected image shape: {test_image.shape}"
    )

if int(test_label) not in INDEX_TO_CLASS:
    raise ValueError(
        f"Unexpected class index: {int(test_label)}"
    )

if (
    INDEX_TO_CLASS[int(test_label)]
    != test_record["class"]
):
    raise ValueError(
        "The numeric label does not match "
        "the original class label."
    )

if not bool(
    tf.reduce_all(
        tf.math.is_finite(test_image)
    )
):
    raise ValueError(
        "The preprocessed image contains "
        "NaN or infinite values."
    )

print("\nReusable ResNet50 image loader passed.")

--- Class mapping ---
0 -> glioma
1 -> meningioma
2 -> notumor
3 -> pituitary

--- Label distribution ---


,class,class_index,number_of_images
0,glioma,0,1400
1,meningioma,1,1400
2,notumor,2,1400
3,pituitary,3,1400



--- Reusable loader test ---
Image shape: (224, 224, 3)
Image data type: <dtype: 'float32'>
Numeric label: 0
Decoded class: glioma
Expected class: glioma
All image values finite: True

Reusable ResNet50 image loader passed.


In [ ]:
# Create reusable TensorFlow datasets for one cross-validation fold


# ---------------------------------------------------------
# 1. Local data-pipeline settings
# ---------------------------------------------------------

# Keep parallelism modest while Random Forest tuning
# is still using four CPU workers.
DATA_PIPELINE_PARALLEL_CALLS = 2
PREFETCH_BATCHES = 1


# ---------------------------------------------------------
# 2. Define the dataset-construction function
# ---------------------------------------------------------

def create_resnet50_dataset(
    dataframe,
    training,
    batch_size=BATCH_SIZE
):
    """
    Create a TensorFlow dataset from image paths and labels.

    Training datasets are shuffled.
    Validation datasets retain their fixed order.
    """

    # Extracting all image paths
    image_paths = (
        dataframe["image_path"]
        .astype(str)
        .to_numpy()
    )
    
    # Extracting all image class indices
    class_indices = (
        dataframe["class_index"]
        .astype(np.int32)
        .to_numpy()
    )

    # Creating a TensorFlow dataset
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            image_paths,
            class_indices
        )
    )

    
    if training:     # Checking if the function creates a training dataset
        dataset = dataset.shuffle(  # Shuffling the dataset
            buffer_size=len(dataframe),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    # Loading and preparing the images for ResNet50
    dataset = dataset.map(
        load_and_preprocess_resnet50_image,
        num_parallel_calls=DATA_PIPELINE_PARALLEL_CALLS,
        deterministic=True
    )

# Grouping individual dataset images into batches
    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(
        PREFETCH_BATCHES
    )

    return dataset


# ---------------------------------------------------------
# 3. Create the Fold 1 training and validation tables
# ---------------------------------------------------------

VALIDATION_FOLD = 1

# Creating the training fold
fold_1_training_df = (
    folds_df[
        folds_df["fold"] != VALIDATION_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)

# Creating the validation fold
fold_1_validation_df = (
    folds_df[
        folds_df["fold"] == VALIDATION_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 4. Validate the split sizes and class distributions
# ---------------------------------------------------------

print("--- Fold 1 split ---")
print(
    "Training images:",
    len(fold_1_training_df)
)
print(
    "Validation images:",
    len(fold_1_validation_df)
)

print("\n--- Training class distribution ---")
print(
    fold_1_training_df["class"]
    .value_counts()
    .sort_index()
)

print("\n--- Validation class distribution ---")
print(
    fold_1_validation_df["class"]
    .value_counts()
    .sort_index()
)


if len(fold_1_training_df) != 4480:
    raise ValueError(
        "Fold 1 should contain 4,480 training images."
    )

if len(fold_1_validation_df) != 1120:
    raise ValueError(
        "Fold 1 should contain 1,120 validation images."
    )


# ---------------------------------------------------------
# 5. Create the TensorFlow datasets
# ---------------------------------------------------------

fold_1_training_dataset = (
    create_resnet50_dataset(
        fold_1_training_df,
        training=True
    )
)

fold_1_validation_dataset = (
    create_resnet50_dataset(
        fold_1_validation_df,
        training=False
    )
)


# ---------------------------------------------------------
# 6. Inspect one batch from each dataset
# ---------------------------------------------------------

training_images, training_labels = next(
    iter(fold_1_training_dataset)
)

validation_images, validation_labels = next(
    iter(fold_1_validation_dataset)
)


print("\n--- Training batch ---")
print("Image shape:", training_images.shape)
print("Image dtype:", training_images.dtype)
print("Label shape:", training_labels.shape)
print("Label dtype:", training_labels.dtype)
print(
    "Labels:",
    training_labels.numpy().tolist()
)
print(
    "All image values finite:",
    bool(
        tf.reduce_all(
            tf.math.is_finite(training_images)
        )
    )
)


print("\n--- Validation batch ---")
print("Image shape:", validation_images.shape)
print("Image dtype:", validation_images.dtype)
print("Label shape:", validation_labels.shape)
print("Label dtype:", validation_labels.dtype)
print(
    "Labels:",
    validation_labels.numpy().tolist()
)
print(
    "All image values finite:",
    bool(
        tf.reduce_all(
            tf.math.is_finite(validation_images)
        )
    )
)


# ---------------------------------------------------------
# 7. Final batch checks
# ---------------------------------------------------------

expected_batch_shape = (
    BATCH_SIZE,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    IMAGE_CHANNELS
)

if training_images.shape != expected_batch_shape:
    raise ValueError(
        "Unexpected training batch shape: "
        f"{training_images.shape}"
    )

if validation_images.shape != expected_batch_shape:
    raise ValueError(
        "Unexpected validation batch shape: "
        f"{validation_images.shape}"
    )

if not bool(
    tf.reduce_all(
        tf.math.is_finite(training_images)
    )
):
    raise ValueError(
        "The training batch contains invalid values."
    )

if not bool(
    tf.reduce_all(
        tf.math.is_finite(validation_images)
    )
):
    raise ValueError(
        "The validation batch contains invalid values."
    )

print("\nFold 1 TensorFlow datasets passed.")

--- Fold 1 split ---
Training images: 4480
Validation images: 1120

--- Training class distribution ---
class
glioma        1120
meningioma    1120
notumor       1120
pituitary     1120
Name: count, dtype: int64

--- Validation class distribution ---
class
glioma        280
meningioma    280
notumor       280
pituitary     280
Name: count, dtype: int64

--- Training batch ---
Image shape: (16, 224, 224, 3)
Image dtype: <dtype: 'float32'>
Label shape: (16,)
Label dtype: <dtype: 'int32'>
Labels: [2, 2, 2, 3, 3, 2, 2, 1, 2, 3, 1, 3, 2, 2, 1, 1]
All image values finite: True

--- Validation batch ---
Image shape: (16, 224, 224, 3)
Image dtype: <dtype: 'float32'>
Label shape: (16,)
Label dtype: <dtype: 'int32'>
Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
All image values finite: True

Fold 1 TensorFlow datasets passed.
